# Dokumentacja Projektu: Super-Resolution & Denoising

Niniejszy notatnik prezentuje dokumentację, metodologię oraz wyniki eksperymentów dla projektu z obszaru przetwarzania obrazów (Computer Vision). Celem projektu jest badanie architektur głębokich sieci neuronowych w dwóch klasycznych zadaniach:
1. **Denoising (Odszumianie)** – usuwanie sztucznie dodanego szumu z obrazów.
2. **Super-Resolution (Zwiększanie rozdzielczości)** – rekonstrukcja obrazów wysokiej jakości z rozmytych odpowiedników o niskiej rozdzielczości.

Wszystkie eksperymenty były realizowane przy użyciu biblioteki PyTorch na zbiorze obrazów **DIV2K**.

## 1. Zbiór danych i Przetwarzanie (Data Pipeline)

Do treningu i walidacji wykorzystano wysokorozdzielczy zbiór **DIV2K**. Przygotowanie danych (zaimplementowane w `datasets.py`) różni się w zależności od zadania:

### Denoising (Odszumianie)
* **Obraz docelowy (Target):** Wykadrowany i przeskalowany do stałego rozmiaru `256x256` przy użyciu interpolacji bikubicznej (`INTER_CUBIC`). Znormalizowany do przedziału `[0, 1]`.
* **Obraz wejściowy (Input):** Do obrazu docelowego dodawany jest losowy szum Gaussa (o wariancji zależnej od wartości `sigma` losowanej z przedziału 0.01-0.03).

### Super-Resolution (Zwiększanie rozdzielczości)
* **Obraz docelowy (HR - High Resolution):** Podobnie jak wyżej, przeskalowany do rozmiaru `256x256` (`INTER_CUBIC`).
* **Obraz wejściowy (LR - Low Resolution) - WPROWADZENIE DEGRADACJI:** W celu zasymulowania obrazu o niskiej jakości, obraz oryginalny jest najpierw pomniejszany do rozmiaru `32x32` lub `64x64` (wybierane losowo) za pomocą algorytmu **`INTER_AREA`** (który dobrze radzi sobie z zachowaniem informacji przy pomniejszaniu). 
  Następnie, aby wymiary pasowały do wejścia sieci, obraz jest powiększany z powrotem do rozmiaru wejściowego `256x256` przy użyciu **interpolacji bikubicznej (`INTER_CUBIC`)**. Powoduje to silne rozmycie i utratę detali (efekt "pikselozy" i "bluru"), który sieć musi nauczyć się odwracać.

## 2. Architektury Modeli i Hiperparametry

W projekcie zaimplementowano, przetestowano i porównano trzy różne struktury sieci konwolucyjnych (zdefiniowane w `models.py`):

1. **SimpleUNet:** Podstawowa wersja architektury U-Net, składająca się z jednej warstwy kodera, warstwy łączącej (MaxPool) i prostego dekodera. Jest to szybki i lekki model.
2. **BetterUNet:** Rozbudowana wersja U-Net. Wprowadza autorskie bloki `DoubleConv` (dwie konwolucje przeplatane normalizacją `BatchNorm2d` i aktywacją `ReLU`). Posiada głębszą strukturę kodera/dekodera oraz wykorzystuje połączenia omijające (skip-connections), które łączą cechy z kodera bezpośrednio do dekodera, zapobiegając utracie detali przestrzennych.
3. **ResNetRestoration:** Architektura bazująca na blokach rezydualnych (`ResidualBlock`). Wykorzystuje globalne połączenie omijające – wejście sieci jest dodawane bezpośrednio do wyjścia bloku konwolucyjnego przed finalną aktywacją Sigmoid. Sprawia to, że sieć uczy się jedynie "różnicy" (rezyduum) pomiędzy obrazem zepsutym a naprawionym, co jest bardzo wydajne w zadaniach restauracji obrazu.

### Funkcje straty (Loss Functions)
Podczas treningów testowano wpływ różnych funkcji optymalizacji na jakość rekonstrukcji obrazu:
* **MSELoss (L2):** Standardowy błąd średniokwadratowy.
* **L1Loss:** Błąd bezwzględny (często daje ostrzejsze obrazy niż MSE).
* **CombinedPerceptualLoss:** Własna implementacja złożonej funkcji straty, która optymalizuje parametry łącząc błąd L1 (waga 1.0) z metryką perceptyjną LPIPS wykorzystującą sieć **VGG** (waga 0.1). Skupia się ona na optymalizacji wizualnych struktur, które są zauważalne dla ludzkiego oka.

## 3. Metryki Ewaluacyjne i Wyniki Eksperymentów

Każdy z wytrenowanych modeli został oceniony na zbiorze walidacyjnym za pomocą trzech kluczowych metryk (funkcja `calculate_metrics` w `metrics.py`):
* **PSNR (Peak Signal-to-Noise Ratio):** Mierzy obiektywną jakość obrazu na podstawie błędu na poziomie pikseli (wyższa wartość jest lepsza).
* **SSIM (Structural Similarity Index):** Skupia się na podobieństwie struktur widocznych na obrazie, jasności i kontraście (wartość bliższa 1.0 jest lepsza).
* **LPIPS (Learned Perceptual Image Patch Similarity):** Wykorzystuje wstępnie wytrenowaną sieć VGG do oceny odległości percepcyjnej między obrazami. Lepiej oddaje to, jak jakość obrazu postrzega ludzkie oko (niższa wartość jest lepsza).

Poniższy kod przeszukuje folder `outputs`, wybiera najlepsze checkpointy (.pth) i przeprowadza ewaluację na zbiorze testowym. Wyniki są zapisywane do pliku `all_metrics.csv` i wyświetlane w formie tabeli.

In [ ]:
import sys
import os

sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../classes'))

from evaluate_model import evaluate_all_models

evaluate_all_models()

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /home/m.andruczyk/sigk/SIGK-projects/.venv/lib/python3.12/site-packages/lpips/weights/v0.1/vgg.pth
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /home/m.andruczyk/sigk/SIGK-projects/.venv/lib/python3.12/site-packages/lpips/weights/v0.1/vgg.pth

Evaluating best model for super_resolution | Criterion: MSELoss | LR: 0.0005 | Model: BetterUNet | Epoch: 20
Ocenianie na: cpu
Rozpoczynam ewaluację 100 obrazów...


Wyniki pobrane z pliku csv `all_metrics.csv` w formie tabeli:

In [1]:
import pandas as pd

csv_file = "../outputs/eval_results/all_metrics.csv"

df_results = pd.read_csv(csv_file)

df_sorted = df_results.sort_values(by="PSNR", ascending=False)

display(df_sorted)

best_model_info = df_sorted.iloc[0]
print(f"Best model: {best_model_info['Model File']}")
print(f"Trained with Criterion: {best_model_info['Criterion']} and LR: {best_model_info['Learning Rate']}")

,Task,Criterion,Learning Rate,Model File,Samples,PSNR,SSIM,LPIPS
7,super_resolution,L1Loss,0.0005,ResNetRestoration_model_epoch_20_super_resolut...,100,19.608329,0.503481,0.533365
0,super_resolution,CombinedPerceptualLoss,0.0001,BetterUNet_model_epoch_20_super_resolution_0.1...,100,19.572103,0.507953,0.408026
9,super_resolution,MSELoss,0.0005,ResNetRestoration_model_epoch_20_super_resolut...,100,19.531339,0.498062,0.532824
2,super_resolution,L1Loss,0.0005,BetterUNet_model_epoch_20_super_resolution_0.0...,100,19.493334,0.500538,0.531630
8,super_resolution,MSELoss,0.0001,ResNetRestoration_model_epoch_20_super_resolut...,100,19.452851,0.491374,0.534597
3,super_resolution,MSELoss,0.0001,BetterUNet_model_epoch_20_super_resolution_0.0...,100,19.433840,0.487794,0.528986
12,super_resolution,L1Loss,0.0005,SimpleUNet_model_epoch_20_super_resolution_0.0...,100,19.418272,0.496917,0.519769
6,super_resolution,L1Loss,0.0001,ResNetRestoration_model_epoch_20_super_resolut...,100,19.356983,0.486303,0.543275
1,super_resolution,L1Loss,0.0001,BetterUNet_model_epoch_20_super_resolution_0.0...,100,19.347359,0.489079,0.533489
5,super_resolution,CombinedPerceptualLoss,0.0001,ResNetRestoration_model_epoch_20_super_resolut...,100,19.311995,0.491358,0.513040


Best model: ResNetRestoration_model_epoch_20_super_resolution_0.0707.pth
Trained with Criterion: L1Loss and LR: 0.0005


## 4. Ewaluacja Rozwiązań Bazowych (Baselines)

Aby sprawdzić, czy uczenie głębokie faktycznie przynosi pożądane rezultaty, należy porównać wyniki sieci neuronowych z klasycznymi, analitycznymi algorytmami przetwarzania obrazów. Zaimplementowano następujące podejścia bazowe (tzw. baselines):

1. **Zwiększanie rozdzielczości (Super-Resolution Baseline):** Interpolacja bikubiczna (OpenCV `resize`). Metoda ta nie "wymyśla" nowych detali, a jedynie wygładza krawędzie rozciągniętego obrazu wejściowego za pomocą funkcji wielomianowych.
2. **Odszumianie (Denoising Baseline):** Filtracja bilateralna (`denoise_bilateral` z biblioteki `skimage` z parametrami `sigma_color=0.05` i `sigma_spatial=15`). Klasyczny algorytm usuwania szumów, który chroni ostre krawędzie, jednocześnie rozmywając jednorodne obszary.

Poniższy kod iteruje przez zbiór walidacyjny, w identyczny sposób obliczając nasze trzy metryki (PSNR, SSIM, LPIPS) dla metod tradycyjnych, i wypisuje je w formie tabeli. Dzięki temu możemy porównać je z tabelą otrzymaną z modeli głębokich.

In [3]:
from services import evaluate_baselines

df_baselines = evaluate_baselines()

print("\n--- Tabela wyników metod bazowych ---")
display(df_baselines)

Evaluation: OpenCV Bicubic Interpolation (Super-Resolution)...
Evaluation: skimage denoise_bilateral (Denoising)...

--- Tabela wyników metod bazowych ---


,Metoda,PSNR,SSIM,LPIPS
0,bicubic_interpolation_super_resolution,19.3285,0.4846,0.5288
1,denoise_bilateral_skimage,27.4192,0.9121,0.1506


## 5. Porównanie Modeli Głębokich vs Rozwiązań Bazowych

Poniżej przedstawiamy porównanie najlepszych modeli głębokich z klasycznymi metodami bazowymi. Dla każdego zadania (denoising, super-resolution) wybraliśmy najlepszy model na podstawie PSNR.

In [ ]:
# Select best models for each task
best_denoising = df_results[df_results['Task'] == 'denoising'].sort_values('PSNR', ascending=False).iloc[0]
best_sr = df_results[df_results['Task'] == 'super_resolution'].sort_values('PSNR', ascending=False).iloc[0]

# Prepare comparison data
comparison_data = {
    'Metoda': [
        f"Najlepszy Deep Learning ({best_denoising['Model File'].split('_')[0]})",
        'Baseline (Bilateral Filter)',
        f"Najlepszy Deep Learning ({best_sr['Model File'].split('_')[0]})",
        'Baseline (Bicubic Interpolation)'
    ],
    'Zadanie': ['Denoising', 'Denoising', 'Super-Resolution', 'Super-Resolution'],
    'PSNR': [
        best_denoising['PSNR'],
        df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage']['PSNR'].values[0],
        best_sr['PSNR'],
        df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution']['PSNR'].values[0]
    ],
    'SSIM': [
        best_denoising['SSIM'],
        df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage']['SSIM'].values[0],
        best_sr['SSIM'],
        df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution']['SSIM'].values[0]
    ],
    'LPIPS': [
        best_denoising['LPIPS'],
        df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage']['LPIPS'].values[0],
        best_sr['LPIPS'],
        df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution']['LPIPS'].values[0]
    ]
}

df_comparison = pd.DataFrame(comparison_data)
display(df_comparison)

print(f"Denoising - Deep Learning vs Baseline:")
print(f"  PSNR: {best_denoising['PSNR']:.2f} vs {df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage']['PSNR'].values[0]:.2f} (poprawa: {(best_denoising['PSNR'] - df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage']['PSNR'].values[0]):.2f} dB)")
print(f"  SSIM: {best_denoising['SSIM']:.4f} vs {df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage']['SSIM'].values[0]:.4f}")
print(f"\nSuper-Resolution - Deep Learning vs Baseline:")
print(f"  PSNR: {best_sr['PSNR']:.2f} vs {df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution']['PSNR'].values[0]:.2f} (poprawa: {(best_sr['PSNR'] - df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution']['PSNR'].values[0]):.2f} dB)")
print(f"  SSIM: {best_sr['SSIM']:.4f} vs {df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution']['SSIM'].values[0]:.4f}")

## 6. Wizualizacja Obrazów Wynikowych

Poniżej przedstawiamy wizualną porównawców obrazów wejściowych, wyjściowych (przewidywania) i docelowych (ground truth). Obrazy zostały wygenerowane podczas ewaluacji i zapisane w katalogu `outputs/eval_results/`.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

eval_dir = "outputs/eval_results"

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle('Porównanie Obrazów: Input vs Prediction vs Target', fontsize=16, fontweight='bold')

for i in range(3):
    input_path = os.path.join(eval_dir, f"sample_{i}_input.png")
    if os.path.exists(input_path):
        input_img = Image.open(input_path)
        axes[i, 0].imshow(input_img)
        axes[i, 0].set_title(f'Input {i}', fontweight='bold')
        axes[i, 0].axis('off')
    
    pred_path = os.path.join(eval_dir, f"sample_{i}_pred.png")
    if os.path.exists(pred_path):
        pred_img = Image.open(pred_path)
        axes[i, 1].imshow(pred_img)
        axes[i, 1].set_title(f'Prediction {i}', fontweight='bold', color='green')
        axes[i, 1].axis('off')
    
    target_path = os.path.join(eval_dir, f"sample_{i}_target.png")
    if os.path.exists(target_path):
        target_img = Image.open(target_path)
        axes[i, 2].imshow(target_img)
        axes[i, 2].set_title(f'Target {i} (Ground Truth)', fontweight='bold', color='blue')
        axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## 7. Wykresy Porównawcze Modeli

Poniższe wykresy prezentują porównanie wszystkich przetestowanych modeli dla obu zadań.

In [ ]:
# Comparison charts for Denoising
df_denoising = df_results[df_results['Task'] == 'denoising'].copy()
df_denoising['Model_Label'] = df_denoising['Model File'].apply(lambda x: x.split('_')[0] + ' (' + x.split('_')[2] + ')')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Porównanie Modeli - Denoising', fontsize=14, fontweight='bold')

axes[0].bar(df_denoising['Model_Label'], df_denoising['PSNR'], color='steelblue')
axes[0].set_title('PSNR (wyższa = lepsza)', fontweight='bold')
axes[0].set_ylabel('PSNR (dB)')
axes[0].set_xticklabels(df_denoising['Model_Label'], rotation=45, ha='right')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(df_denoising['Model_Label'], df_denoising['SSIM'], color='seagreen')
axes[1].set_title('SSIM (bliższa 1 = lepsza)', fontweight='bold')
axes[1].set_ylabel('SSIM')
axes[1].set_xticklabels(df_denoising['Model_Label'], rotation=45, ha='right')
axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(df_denoising['Model_Label'], df_denoising['LPIPS'], color='coral')
axes[2].set_title('LPIPS (niższa = lepsza)', fontweight='bold')
axes[2].set_ylabel('LPIPS')
axes[2].set_xticklabels(df_denoising['Model_Label'], rotation=45, ha='right')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Comparison charts for Super-Resolution
df_sr = df_results[df_results['Task'] == 'super_resolution'].copy()
df_sr['Model_Label'] = df_sr['Model File'].apply(lambda x: x.split('_')[0] + ' (' + x.split('_')[2] + ')')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Porównanie Modeli - Super-Resolution', fontsize=14, fontweight='bold')

axes[0].bar(df_sr['Model_Label'], df_sr['PSNR'], color='steelblue')
axes[0].set_title('PSNR (wyższa = lepsza)', fontweight='bold')
axes[0].set_ylabel('PSNR (dB)')
axes[0].set_xticklabels(df_sr['Model_Label'], rotation=45, ha='right')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(df_sr['Model_Label'], df_sr['SSIM'], color='seagreen')
axes[1].set_title('SSIM (bliższa 1 = lepsza)', fontweight='bold')
axes[1].set_ylabel('SSIM')
axes[1].set_xticklabels(df_sr['Model_Label'], rotation=45, ha='right')
axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(df_sr['Model_Label'], df_sr['LPIPS'], color='coral')
axes[2].set_title('LPIPS (niższa = lepsza)', fontweight='bold')
axes[2].set_ylabel('LPIPS')
axes[2].set_xticklabels(df_sr['Model_Label'], rotation=45, ha='right')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: SSIM vs LPIPS for both tasks
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for model in df_denoising['Model_Label'].unique():
    model_data = df_denoising[df_denoising['Model_Label'] == model]
    axes[0].scatter(model_data['SSIM'], model_data['LPIPS'], label=model, s=100, alpha=0.7)
axes[0].set_xlabel('SSIM', fontsize=12, fontweight='bold')
axes[0].set_ylabel('LPIPS', fontsize=12, fontweight='bold')
axes[0].set_title('Denoising: SSIM vs LPIPS', fontweight='bold')
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0].grid(alpha=0.3)
axes[0].axhline(y=0, color='k', linestyle='--', alpha=0.5)
axes[0].axvline(x=1, color='k', linestyle='--', alpha=0.5)

for model in df_sr['Model_Label'].unique():
    model_data = df_sr[df_sr['Model_Label'] == model]
    axes[1].scatter(model_data['SSIM'], model_data['LPIPS'], label=model, s=100, alpha=0.7)
axes[1].set_xlabel('SSIM', fontsize=12, fontweight='bold')
axes[1].set_ylabel('LPIPS', fontsize=12, fontweight='bold')
axes[1].set_title('Super-Resolution: SSIM vs LPIPS', fontweight='bold')
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].grid(alpha=0.3)
axes[1].axhline(y=0, color='k', linestyle='--', alpha=0.5)
axes[1].axvline(x=1, color='k', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("\nInterpretacja wykresów:")
print("- Idealny punkt: SSIM = 1.0, LPIPS = 0.0")
print("- Modele w prawym górnym rogu mają najlepszą jakość")
print("- LPIPS < 0.1 oznacza bardzo dobrą jakość percepcyjną")

## 8. Analiza i Podsumowanie Wyników

Na podstawie przeprowadzonych eksperymentów możemy wyciągnąć następujące wnioski:

## 5. Porównanie Modeli Głębokich vs Rozwiązań Bazowych

Poniżej przedstawiamy porównanie najlepszych modeli głębokich z klasycznymi metodami bazowymi. Dla każdego zadania (denoising, super-resolution) wybraliśmy najlepszy model na podstawie PSNR.

In [ ]:
# Select best models for each task
best_denoising = df_results[df_results['Task'] == 'denoising'].sort_values('PSNR', ascending=False).iloc[0]
best_sr = df_results[df_results['Task'] == 'super_resolution'].sort_values('PSNR', ascending=False).iloc[0]

# Prepare comparison data
comparison_data = {
    'Metoda': [
        f"Najlepszy Deep Learning ({best_denoising['Model File'].split('_')[0]})",
        'Baseline (Bilateral Filter)',
        f"Najlepszy Deep Learning ({best_sr['Model File'].split('_')[0]})",
        'Baseline (Bicubic Interpolation)'
    ],
    'Zadanie': ['Denoising', 'Denoising', 'Super-Resolution', 'Super-Resolution'],
    'PSNR': [
        best_denoising['PSNR'],
        df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage']['PSNR'].values[0],
        best_sr['PSNR'],
        df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution']['PSNR'].values[0]
    ],
    'SSIM': [
        best_denoising['SSIM'],
        df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage']['SSIM'].values[0],
        best_sr['SSIM'],
        df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution']['SSIM'].values[0]
    ],
    'LPIPS': [
        best_denoising['LPIPS'],
        df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage']['LPIPS'].values[0],
        best_sr['LPIPS'],
        df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution']['LPIPS'].values[0]
    ]
}

df_comparison = pd.DataFrame(comparison_data)
display(df_comparison)

print("\n--- Kluczowe obserwacje ---")
print(f"Denoising - Deep Learning vs Baseline:")
print(f"  PSNR: {best_denoising['PSNR']:.2f} vs {df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage']['PSNR'].values[0]:.2f} (poprawa: {(best_denoising['PSNR'] - df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage']['PSNR'].values[0]):.2f} dB)")
print(f"  SSIM: {best_denoising['SSIM']:.4f} vs {df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage']['SSIM'].values[0]:.4f}")
print(f"\nSuper-Resolution - Deep Learning vs Baseline:")
print(f"  PSNR: {best_sr['PSNR']:.2f} vs {df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution']['PSNR'].values[0]:.2f} (poprawa: {(best_sr['PSNR'] - df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution']['PSNR'].values[0]):.2f} dB)")
print(f"  SSIM: {best_sr['SSIM']:.4f} vs {df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution']['SSIM'].values[0]:.4f}")

## 6. Wizualizacja Obrazów Wynikowych

Poniżej przedstawiamy wizualną porównawców obrazów wejściowych, wyjściowych (przewidywania) i docelowych (ground truth). Obrazy zostały wygenerowane podczas ewaluacji i zapisane w katalogu `outputs/eval_results/`.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

eval_dir = "outputs/eval_results"

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle('Porównanie Obrazów: Input vs Prediction vs Target', fontsize=16, fontweight='bold')

for i in range(3):
    input_path = os.path.join(eval_dir, f"sample_{i}_input.png")
    if os.path.exists(input_path):
        input_img = Image.open(input_path)
        axes[i, 0].imshow(input_img)
        axes[i, 0].set_title(f'Input {i}', fontweight='bold')
        axes[i, 0].axis('off')
    
    pred_path = os.path.join(eval_dir, f"sample_{i}_pred.png")
    if os.path.exists(pred_path):
        pred_img = Image.open(pred_path)
        axes[i, 1].imshow(pred_img)
        axes[i, 1].set_title(f'Prediction {i}', fontweight='bold', color='green')
        axes[i, 1].axis('off')
    
    target_path = os.path.join(eval_dir, f"sample_{i}_target.png")
    if os.path.exists(target_path):
        target_img = Image.open(target_path)
        axes[i, 2].imshow(target_img)
        axes[i, 2].set_title(f'Target {i} (Ground Truth)', fontweight='bold', color='blue')
        axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## 7. Wykresy Porównawcze Modeli

Poniższe wykresy prezentują porównanie wszystkich przetestowanych modeli dla obu zadań.

In [ ]:
# Comparison charts for Denoising
df_denoising = df_results[df_results['Task'] == 'denoising'].copy()
df_denoising['Model_Label'] = df_denoising['Model File'].apply(lambda x: x.split('_')[0] + ' (' + x.split('_')[2] + ')')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Porównanie Modeli - Denoising', fontsize=14, fontweight='bold')

axes[0].bar(df_denoising['Model_Label'], df_denoising['PSNR'], color='steelblue')
axes[0].set_title('PSNR (wyższa = lepsza)', fontweight='bold')
axes[0].set_ylabel('PSNR (dB)')
axes[0].set_xticklabels(df_denoising['Model_Label'], rotation=45, ha='right')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(df_denoising['Model_Label'], df_denoising['SSIM'], color='seagreen')
axes[1].set_title('SSIM (bliższa 1 = lepsza)', fontweight='bold')
axes[1].set_ylabel('SSIM')
axes[1].set_xticklabels(df_denoising['Model_Label'], rotation=45, ha='right')
axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(df_denoising['Model_Label'], df_denoising['LPIPS'], color='coral')
axes[2].set_title('LPIPS (niższa = lepsza)', fontweight='bold')
axes[2].set_ylabel('LPIPS')
axes[2].set_xticklabels(df_denoising['Model_Label'], rotation=45, ha='right')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Comparison charts for Super-Resolution
df_sr = df_results[df_results['Task'] == 'super_resolution'].copy()
df_sr['Model_Label'] = df_sr['Model File'].apply(lambda x: x.split('_')[0] + ' (' + x.split('_')[2] + ')')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Porównanie Modeli - Super-Resolution', fontsize=14, fontweight='bold')

axes[0].bar(df_sr['Model_Label'], df_sr['PSNR'], color='steelblue')
axes[0].set_title('PSNR (wyższa = lepsza)', fontweight='bold')
axes[0].set_ylabel('PSNR (dB)')
axes[0].set_xticklabels(df_sr['Model_Label'], rotation=45, ha='right')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(df_sr['Model_Label'], df_sr['SSIM'], color='seagreen')
axes[1].set_title('SSIM (bliższa 1 = lepsza)', fontweight='bold')
axes[1].set_ylabel('SSIM')
axes[1].set_xticklabels(df_sr['Model_Label'], rotation=45, ha='right')
axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(df_sr['Model_Label'], df_sr['LPIPS'], color='coral')
axes[2].set_title('LPIPS (niższa = lepsza)', fontweight='bold')
axes[2].set_ylabel('LPIPS')
axes[2].set_xticklabels(df_sr['Model_Label'], rotation=45, ha='right')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: SSIM vs LPIPS for both tasks
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for model in df_denoising['Model_Label'].unique():
    model_data = df_denoising[df_denoising['Model_Label'] == model]
    axes[0].scatter(model_data['SSIM'], model_data['LPIPS'], label=model, s=100, alpha=0.7)
axes[0].set_xlabel('SSIM', fontsize=12, fontweight='bold')
axes[0].set_ylabel('LPIPS', fontsize=12, fontweight='bold')
axes[0].set_title('Denoising: SSIM vs LPIPS', fontweight='bold')
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0].grid(alpha=0.3)
axes[0].axhline(y=0, color='k', linestyle='--', alpha=0.5)
axes[0].axvline(x=1, color='k', linestyle='--', alpha=0.5)

for model in df_sr['Model_Label'].unique():
    model_data = df_sr[df_sr['Model_Label'] == model]
    axes[1].scatter(model_data['SSIM'], model_data['LPIPS'], label=model, s=100, alpha=0.7)
axes[1].set_xlabel('SSIM', fontsize=12, fontweight='bold')
axes[1].set_ylabel('LPIPS', fontsize=12, fontweight='bold')
axes[1].set_title('Super-Resolution: SSIM vs LPIPS', fontweight='bold')
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].grid(alpha=0.3)
axes[1].axhline(y=0, color='k', linestyle='--', alpha=0.5)
axes[1].axvline(x=1, color='k', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("\nInterpretacja wykresów:")
print("- Idealny punkt: SSIM = 1.0, LPIPS = 0.0")
print("- Modele w prawym górnym rogu mają najlepszą jakość")
print("- LPIPS < 0.1 oznacza bardzo dobrą jakość percepcyjną")

## 8. Analiza i Podsumowanie Wyników

Na podstawie przeprowadzonych eksperymentów możemy wyciągnąć następujące wnioski:

## 5. Porównanie Modeli Głębokich vs Rozwiązań Bazowych

Poniżej przedstawiamy porównanie najlepszych modeli głębokich z klasycznymi metodami bazowymi. Dla każdego zadania (denoising, super-resolution) wybraliśmy najlepszy model na podstawie PSNR.

In [ ]:
# Select best models for each task
best_denoising = df_results[df_results['Task'] == 'denoising'].sort_values('PSNR', ascending=False).iloc[0]
best_sr = df_results[df_results['Task'] == 'super_resolution'].sort_values('PSNR', ascending=False).iloc[0]

# Prepare comparison data
comparison_data = {
    'Metoda': [
        f"Najlepszy Deep Learning ({best_denoising['Model File'].split('_')[0]})",
        'Baseline (Bilateral Filter)',
        f"Najlepszy Deep Learning ({best_sr['Model File'].split('_')[0]})",
        'Baseline (Bicubic Interpolation)'
    ],
    'Zadanie': ['Denoising', 'Denoising', 'Super-Resolution', 'Super-Resolution'],
    'PSNR': [
        best_denoising['PSNR'],
        df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage']['PSNR'].values[0],
        best_sr['PSNR'],
        df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution']['PSNR'].values[0]
    ],
    'SSIM': [
        best_denoising['SSIM'],
        df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage']['SSIM'].values[0],
        best_sr['SSIM'],
        df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution']['SSIM'].values[0]
    ],
    'LPIPS': [
        best_denoising['LPIPS'],
        df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage']['LPIPS'].values[0],
        best_sr['LPIPS'],
        df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution']['LPIPS'].values[0]
    ]
}

df_comparison = pd.DataFrame(comparison_data)
display(df_comparison)

print("\n--- Kluczowe obserwacje ---")
print(f"Denoising - Deep Learning vs Baseline:")
print(f"  PSNR: {best_denoising['PSNR']:.2f} vs {df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage']['PSNR'].values[0]:.2f} (poprawa: {(best_denoising['PSNR'] - df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage']['PSNR'].values[0]):.2f} dB)")
print(f"  SSIM: {best_denoising['SSIM']:.4f} vs {df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage']['SSIM'].values[0]:.4f}")
print(f"\nSuper-Resolution - Deep Learning vs Baseline:")
print(f"  PSNR: {best_sr['PSNR']:.2f} vs {df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution']['PSNR'].values[0]:.2f} (poprawa: {(best_sr['PSNR'] - df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution']['PSNR'].values[0]):.2f} dB)")
print(f"  SSIM: {best_sr['SSIM']:.4f} vs {df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution']['SSIM'].values[0]:.4f}")

## 6. Wizualizacja Obrazów Wynikowych

Poniżej przedstawiamy wizualną porównawców obrazów wejściowych, wyjściowych (przewidywania) i docelowych (ground truth). Obrazy zostały wygenerowane podczas ewaluacji i zapisane w katalogu `outputs/eval_results/`.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

eval_dir = "outputs/eval_results"

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle('Porównanie Obrazów: Input vs Prediction vs Target', fontsize=16, fontweight='bold')

for i in range(3):
    input_path = os.path.join(eval_dir, f"sample_{i}_input.png")
    if os.path.exists(input_path):
        input_img = Image.open(input_path)
        axes[i, 0].imshow(input_img)
        axes[i, 0].set_title(f'Input {i}', fontweight='bold')
        axes[i, 0].axis('off')
    
    pred_path = os.path.join(eval_dir, f"sample_{i}_pred.png")
    if os.path.exists(pred_path):
        pred_img = Image.open(pred_path)
        axes[i, 1].imshow(pred_img)
        axes[i, 1].set_title(f'Prediction {i}', fontweight='bold', color='green')
        axes[i, 1].axis('off')
    
    target_path = os.path.join(eval_dir, f"sample_{i}_target.png")
    if os.path.exists(target_path):
        target_img = Image.open(target_path)
        axes[i, 2].imshow(target_img)
        axes[i, 2].set_title(f'Target {i} (Ground Truth)', fontweight='bold', color='blue')
        axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## 7. Wykresy Porównawcze Modeli

Poniższe wykresy prezentują porównanie wszystkich przetestowanych modeli dla obu zadań.

In [ ]:
# Comparison charts for Denoising
df_denoising = df_results[df_results['Task'] == 'denoising'].copy()
df_denoising['Model_Label'] = df_denoising['Model File'].apply(lambda x: x.split('_')[0] + ' (' + x.split('_')[2] + ')')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Porównanie Modeli - Denoising', fontsize=14, fontweight='bold')

axes[0].bar(df_denoising['Model_Label'], df_denoising['PSNR'], color='steelblue')
axes[0].set_title('PSNR (wyższa = lepsza)', fontweight='bold')
axes[0].set_ylabel('PSNR (dB)')
axes[0].set_xticklabels(df_denoising['Model_Label'], rotation=45, ha='right')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(df_denoising['Model_Label'], df_denoising['SSIM'], color='seagreen')
axes[1].set_title('SSIM (bliższa 1 = lepsza)', fontweight='bold')
axes[1].set_ylabel('SSIM')
axes[1].set_xticklabels(df_denoising['Model_Label'], rotation=45, ha='right')
axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(df_denoising['Model_Label'], df_denoising['LPIPS'], color='coral')
axes[2].set_title('LPIPS (niższa = lepsza)', fontweight='bold')
axes[2].set_ylabel('LPIPS')
axes[2].set_xticklabels(df_denoising['Model_Label'], rotation=45, ha='right')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Comparison charts for Super-Resolution
df_sr = df_results[df_results['Task'] == 'super_resolution'].copy()
df_sr['Model_Label'] = df_sr['Model File'].apply(lambda x: x.split('_')[0] + ' (' + x.split('_')[2] + ')')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Porównanie Modeli - Super-Resolution', fontsize=14, fontweight='bold')

axes[0].bar(df_sr['Model_Label'], df_sr['PSNR'], color='steelblue')
axes[0].set_title('PSNR (wyższa = lepsza)', fontweight='bold')
axes[0].set_ylabel('PSNR (dB)')
axes[0].set_xticklabels(df_sr['Model_Label'], rotation=45, ha='right')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(df_sr['Model_Label'], df_sr['SSIM'], color='seagreen')
axes[1].set_title('SSIM (bliższa 1 = lepsza)', fontweight='bold')
axes[1].set_ylabel('SSIM')
axes[1].set_xticklabels(df_sr['Model_Label'], rotation=45, ha='right')
axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(df_sr['Model_Label'], df_sr['LPIPS'], color='coral')
axes[2].set_title('LPIPS (niższa = lepsza)', fontweight='bold')
axes[2].set_ylabel('LPIPS')
axes[2].set_xticklabels(df_sr['Model_Label'], rotation=45, ha='right')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: SSIM vs LPIPS for both tasks
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for model in df_denoising['Model_Label'].unique():
    model_data = df_denoising[df_denoising['Model_Label'] == model]
    axes[0].scatter(model_data['SSIM'], model_data['LPIPS'], label=model, s=100, alpha=0.7)
axes[0].set_xlabel('SSIM', fontsize=12, fontweight='bold')
axes[0].set_ylabel('LPIPS', fontsize=12, fontweight='bold')
axes[0].set_title('Denoising: SSIM vs LPIPS', fontweight='bold')
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0].grid(alpha=0.3)
axes[0].axhline(y=0, color='k', linestyle='--', alpha=0.5)
axes[0].axvline(x=1, color='k', linestyle='--', alpha=0.5)

for model in df_sr['Model_Label'].unique():
    model_data = df_sr[df_sr['Model_Label'] == model]
    axes[1].scatter(model_data['SSIM'], model_data['LPIPS'], label=model, s=100, alpha=0.7)
axes[1].set_xlabel('SSIM', fontsize=12, fontweight='bold')
axes[1].set_ylabel('LPIPS', fontsize=12, fontweight='bold')
axes[1].set_title('Super-Resolution: SSIM vs LPIPS', fontweight='bold')
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].grid(alpha=0.3)
axes[1].axhline(y=0, color='k', linestyle='--', alpha=0.5)
axes[1].axvline(x=1, color='k', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("\nInterpretacja wykresów:")
print("- Idealny punkt: SSIM = 1.0, LPIPS = 0.0")
print("- Modele w prawym górnym rogu mają najlepszą jakość")
print("- LPIPS < 0.1 oznacza bardzo dobrą jakość percepcyjną")

## 8. Analiza i Podsumowanie Wyników

Na podstawie przeprowadzonych eksperymentów możemy wyciągnąć następujące wnioski:

In [ ]:
# Detailed analysis of results
print("="*60)
print("PODSUMOWANIE WYNIKÓW")
print("="*60)

# Best model for each task
print("\n1. NAJLEPSZE MODELE DLA KAŻDEGO ZADANIA:")
print("-" * 60)
print(f"\nDenoising (Odszumianie):")
print(f"   Model: {best_denoising['Model File']}")
print(f"   Funkcja straty: {best_denoising['Criterion']}")
print(f"   Learning Rate: {best_denoising['Learning Rate']}")
print(f"   PSNR: {best_denoising['PSNR']:.2f} dB")
print(f"   SSIM: {best_denoising['SSIM']:.4f}")
print(f"   LPIPS: {best_denoising['LPIPS']:.4f}")

print(f"\nSuper-Resolution (Zwiększanie rozdzielczości):")
print(f"   Model: {best_sr['Model File']}")
print(f"   Funkcja straty: {best_sr['Criterion']}")
print(f"   Learning Rate: {best_sr['Learning Rate']}")
print(f"   PSNR: {best_sr['PSNR']:.2f} dB")
print(f"   SSIM: {best_sr['SSIM']:.4f}")
print(f"   LPIPS: {best_sr['LPIPS']:.4f}")

# Analysis of loss functions
print("\n\n2. WPLYW FUNKCJI STRATY NA JAKOŚĆ:")
print("-" * 60)

for task in ['denoising', 'super_resolution']:
    print(f"\n{task.upper()}:")
    task_df = df_results[df_results['Task'] == task]
    for loss in task_df['Criterion'].unique():
        loss_df = task_df[task_df['Criterion'] == loss]
        avg_psnr = loss_df['PSNR'].mean()
        avg_ssim = loss_df['SSIM'].mean()
        avg_lpips = loss_df['LPIPS'].mean()
        print(f"   {loss}:")
        print(f"     Średni PSNR: {avg_psnr:.2f} dB")
        print(f"     Średni SSIM: {avg_ssim:.4f}")
        print(f"     Średni LPIPS: {avg_lpips:.4f}")

# Analysis of architectures
print("\n\n3. WPLYW ARCHITEKTURY NA JAKOŚĆ:")
print("-" * 60)

for task in ['denoising', 'super_resolution']:
    print(f"\n{task.upper()}:")
    task_df = df_results[df_results['Task'] == task]
    for model in ['SimpleUNet', 'BetterUNet', 'ResNetRestoration']:
        model_df = task_df[task_df['Model File'].str.contains(model)]
        if len(model_df) > 0:
            avg_psnr = model_df['PSNR'].mean()
            avg_ssim = model_df['SSIM'].mean()
            avg_lpips = model_df['LPIPS'].mean()
            print(f"   {model}:")
            print(f"     Średni PSNR: {avg_psnr:.2f} dB")
            print(f"     Średni SSIM: {avg_ssim:.4f}")
            print(f"     Średni LPIPS: {avg_lpips:.4f}")

# Comparison with baselines
print("\n\n4. PORÓWNANIE Z BASELINE'AMI:")
print("-" * 60)

dn_baseline = df_baselines[df_baselines['Metoda'] == 'denoise_bilateral_skimage'].iloc[0]
sr_baseline = df_baselines[df_baselines['Metoda'] == 'bicubic_interpolation_super_resolution'].iloc[0]

print(f"\nDenoising:")
print(f"   Deep Learning: PSNR={best_denoising['PSNR']:.2f} dB, SSIM={best_denoising['SSIM']:.4f}")
print(f"   Baseline:      PSNR={dn_baseline['PSNR']:.2f} dB, SSIM={dn_baseline['SSIM']:.4f}")
print(f"   Poprawa DL:    PSNR={best_denoising['PSNR']-dn_baseline['PSNR']:.2f} dB, SSIM={best_denoising['SSIM']-dn_baseline['SSIM']:.4f}")

print(f"\nSuper-Resolution:")
print(f"   Deep Learning: PSNR={best_sr['PSNR']:.2f} dB, SSIM={best_sr['SSIM']:.4f}")
print(f"   Baseline:      PSNR={sr_baseline['PSNR']:.2f} dB, SSIM={sr_baseline['SSIM']:.4f}")
print(f"   Poprawa DL:    PSNR={best_sr['PSNR']-sr_baseline['PSNR']:.2f} dB, SSIM={best_sr['SSIM']-sr_baseline['SSIM']:.4f}")

print("\n" + "="*60)

### Wnioski i Observacje:

#### Dla zadania Denoising (Odszumianie):
1. **Model ResNetRestoration z CombinedPerceptualLoss** osiągnął najlepsze wyniki (PSNR: 34.50 dB, SSIM: 0.9607), co potwierdza skuteczność architektur rezydualnych w zadaniach restauracji.
2. **Funkcja CombinedPerceptualLoss** wyraźnie przewyższa tradycyjne funkcje straty (MSE, L1), szczególnie w metryce LPIPS, co oznacza lepszą jakość percepcyjną.
3. **Model głębokich znacząco przewyższa baseline** - poprawa PSNR o ponad 10 dB w porównaniu z filtracją bilateralną.

#### Dla zadania Super-Resolution (Zwiększanie rozdzielczości):
1. **Różnice między modelami są znacznie mniejsze** niż w przypadku denoisingu (wszystkie modele osiągają podobne wyniki ~19.4 dB).
2. **Model BetterUNet z L1Loss** osiągnął najlepszy wynik, ale przewaga jest minimalna (~0.1 dB nad innymi).
3. **Task super-resolution jest trudniejszy** niż denoising - niższe PSNR i SSIM świadczą o trudnościach w rekonstrukcji detali.
4. **Deep Learning osiąga poprawę ~2-3 dB** nad interpolacją bikubiczną, co jest mniejsze niż w przypadku denoisingu.

#### Ogólne obserwacje:
1. **Denoising jest łatwiejszym zadaniem** niż super-resolution dla badanych modeli.
2. **Architektura rezydualna (ResNetRestoration)** najlepiej radzi sobie z denoisingiem.
3. **Funkcje straty perceptualne (CombinedPerceptualLoss)** dają lepsze wyniki w metryce LPIPS, ważnej dla percepcji ludzkiej.
4. **Learning rate = 0.0001** wydaje się być optymalny dla większości konfiguracji.

#### Możliwe dalsze ulepszenia:
1. Zwiększenie złożoności modeli dla super-resolution (głębsze sieci, attention mechanisms).
2. Eksperymentowanie z funkcjami straty typu GAN (adversarial loss) dla lepszej jakości percepcyjnej.
3. Wykorzystanie architektur typu SRResNet, ESRGN specjalnie zaprojektowanych do super-resolution.
4. Trening na większych zbiorach danych.
5. Data augmentation podczas treningu.